In [ ]:
#imports
import numpy as np
from sklearn.model_selection import train_test_split, KFold
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error

import warnings
warnings.filterwarnings('ignore')



In [ ]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("mohammad2012191/q1-ka-ai-2026")

print("Path to dataset files:", path)

In [ ]:
# Task 1: Write your code here:
import os
import pandas as pd
data_path = os.path.join(path, 'Q1_data.csv')
df = pd.read_csv(data_path)

In [ ]:
# Task 2: Write your code here:
df.head()

In [ ]:
# Task 3: Write your code here:
df.info()

In [ ]:
# Task 4: Write your code here:
df.describe()

In [ ]:
# Task 5: Write your code here:
import matplotlib.pyplot as plt

plt.figure(figsize=(10, 5))
plt.hist(df['Delivery_Time'].dropna(), bins=20, edgecolor='black')
plt.title('Delivery time Distribution')
plt.xlabel('Delivery time')
plt.ylabel('Frequency')
plt.show()

In [ ]:
# Task 1: Write your code here:
df = df.drop(columns=['Order_ID'])
df

In [ ]:
# Task 2: Write your code here:
# Analyze missing values
missing_percentage = (df.isnull().sum() / len(df)) * 100
missing_data = pd.DataFrame({
    'Column': missing_percentage.index,
    'Missing_Percentage': missing_percentage.values
})
missing_data = missing_data[missing_data['Missing_Percentage'] > 0].sort_values('Missing_Percentage', ascending=False)

print("Missing Data Analysis:")
missing_data.head()

In [ ]:
# The number of missing values is small, then we are going to drop those rows
df = df.dropna()
df.info()

In [ ]:
# Task 3: Write your code here:
def check_duplicates(df):
  duplicates = df.duplicated().sum()
  print(f"Number of Duplicate Samples: {duplicates}")
  if duplicates > 0:
    print("Dropping Duplicates...")
    df.drop_duplicates(inplace=True)
    print("Duplicates Dropped.")
  else:
    print("No Duplicate Samples Found.")

check_duplicates(df)
df.info()

In [ ]:
# Task 4: Write your code here:
# Encode categorical columns - converts text to integers
from sklearn.preprocessing import LabelEncoder #import LabelEncoder
categorical_cols = ['Weather', 'Traffic_Level', 'Time_of_Day', 'Vehicle_Type']
for col in categorical_cols:
    encoder = LabelEncoder()
    df[col] = encoder.fit_transform(df[col].astype(str))

df.head()

In [ ]:
# Task 5: Write your code here:
from sklearn.preprocessing import StandardScaler

features = df.columns.drop("Delivery_Time")  # DON'T SCALE THE TARGET

scaler = StandardScaler()
df[features] = scaler.fit_transform(df[features])
df.head()

In [ ]:
# Task 6: Write your code here:


In [ ]:
# Task 1: Write your code here:
X = df.drop('Delivery_Time', axis=1).astype(float)
y = df['Delivery_Time'].astype(float)

In [ ]:
# Task 2,3,4,5: Write your code here:
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error
n_splits = 5 # K
mae_scores = []
# Stratified 5-Fold Cross-Validation, shuffled
skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=42)
model = RandomForestRegressor(n_estimators=100, max_depth=20, random_state=42, n_jobs=-1)


for fold_idx, (train_index, test_index) in enumerate(skf.split(X, y)):
  print(f"\nFold {fold_idx + 1}/{n_splits}")

  # 1. Split data
  X_train, X_test = X.iloc[train_index], X.iloc[test_index]
  y_train, y_test = y.iloc[train_index], y.iloc[test_index]
  model.fit(X_train, y_train) # train
  y_pred = model.predict(X_test) # validate

  # Calculate metric
  mae_scores.append(mean_absolute_error(y_test, y_pred))
mae_scores = np.array(mae_scores)
print(f"MAE:  ${mae_scores.mean():,.2f}")

In [ ]:
# Task 1: Write your code here:
feature_cols = X.columns
feature_importance = pd.DataFrame({
    'feature': feature_cols,
    'importance': model.feature_importances_
}).sort_values('importance', ascending=False)

plt.figure(figsize=(10, 6))
plt.barh(feature_importance['feature'], feature_importance['importance'])
plt.xlabel('Importance')
plt.title('Feature Importance')
plt.gca().invert_yaxis()
plt.show()

In [ ]:
# Task 2: Write your code here:
plt.hist(df['Delivery_Time'])

In [ ]:
# Task Bonus: Write your code here:
n_splits = 5 # K
mae_scores = []
# Stratified 5-Fold Cross-Validation, shuffled
skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=42)
models = {"Random Forest Regressor": RandomForestRegressor(n_estimators=200),
          "Ridge Regression": Ridge(alpha=1.0, max_iter=10000)}

for fold_idx, (train_index, test_index) in enumerate(skf.split(X, y)):
  av = []
  print(f"\nFold {fold_idx + 1}/{n_splits}")

  # 1. Split data
  X_train, X_test = X.iloc[train_index], X.iloc[test_index]
  y_train, y_test = y.iloc[train_index], y.iloc[test_index]
  for model_name, model in models.items():
    print(f"Training {model_name}...")

    # Train
    model.fit(X_train, y_train)

    # Predict
    y_pred = model.predict(X_test)
    av.append(y_pred)


  # Calculate metric
  mae_scores.append(mean_absolute_error(y_test, y_pred))
mae_scores = np.array(mae_scores)
print(f"MAE:  ${mae_scores.mean():,.2f}")